# U8 — Data Cleaning & Preprocessing (Part 2): Lab

**Preparing Data for ML** — encoding · feature scaling · feature engineering · the preprocessing pipeline

_Day 4 · Phase C — Data Engineering & Preparation · From clean data to model-ready data._

#objectives

By the end of this lab you will be able to:

Turn categorical columns into numbers with label and one-hot encoding

Scale features with standardisation (Z-score) and normalisation (Min-Max)

Fit transformers on training data only, to avoid data leakage

Engineer new features that expose the signal more directly

Wrap preprocessing and a model into a single scikit-learn Pipeline

#how to use this lab

Each section has two kinds of cells:

Worked demo cells — run them top to bottom and read the comments to learn the pattern.

LAB EXERCISE cells (marked 🧪) — your turn. Replace each `# YOUR CODE HERE` with working code.

Run cells with **Shift + Enter**. Run the demos before attempting the exercises.

In [1]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.2


In [2]:
# -----------------------------------------------------------
# A SMALL, ALREADY-CLEAN DATASET (Part 1 did the cleaning)
# -----------------------------------------------------------
# Mixed columns: an unordered category (city), an ordered category
# (size), two numeric features on very different scales, and a target.
df = pd.DataFrame({
    'city':   ['pune', 'delhi', 'mumbai', 'pune', 'delhi', 'mumbai', 'pune', 'delhi'],
    'size':   ['small', 'large', 'medium', 'medium', 'small', 'large', 'large', 'small'],
    'age':    [25, 41, 33, 29, 52, 38, 46, 22],
    'income': [38000, 92000, 55000, 47000, 120000, 76000, 88000, 41000],
    'bought': [0, 1, 0, 0, 1, 1, 1, 0],   # target
})
df

,city,size,age,income,bought
0,pune,small,25,38000,0
1,delhi,large,41,92000,1
2,mumbai,medium,33,55000,0
3,pune,medium,29,47000,0
4,delhi,small,52,120000,1
5,mumbai,large,38,76000,1
6,pune,large,46,88000,1
7,delhi,small,22,41000,0


#1. Encoding categorical data

In [5]:
# -----------------------------------------------------------
# 🔹 1A. ONE-HOT ENCODING (for UNORDERED categories)
# -----------------------------------------------------------

# 'city' has no natural order -> one 0/1 column per category
city_ohe = pd.get_dummies(df['city'], prefix='city').astype(int)
city_ohe

,city_delhi,city_mumbai,city_pune
0,0,0,1
1,1,0,0
2,0,1,0
3,0,0,1
4,1,0,0
5,0,1,0
6,0,0,1
7,1,0,0


In [6]:
# -----------------------------------------------------------
# 🔹 1B. LABEL / ORDINAL ENCODING (for ORDERED categories)
# -----------------------------------------------------------

# 'size' has a real order: small < medium < large -> map to 0,1,2
size_order = {'small': 0, 'medium': 1, 'large': 2}
df['size_code'] = df['size'].map(size_order)
df[['size', 'size_code']]

,size,size_code
0,small,0
1,large,2
2,medium,1
3,medium,1
4,small,0
5,large,2
6,large,2
7,small,0


#### 🧪 LAB EXERCISE 1 — Encode the columns

1. One-hot encode `city` again, but this time use `pd.get_dummies` on the **whole DataFrame** (pass `columns=['city']`).
2. Ordinal-encode `size` using `size_order` (you can reuse `size_code`).
3. In a comment, explain why one-hot is wrong for `size` and ordinal is wrong for `city`.

In [27]:
# 1. one-hot encode just the 'city' column of the whole df
# YOUR CODE HERE
city = pd.get_dummies(df['city'], columns = ['city']).astype(int)
print(city)


# 2. ordinal-encode 'size' (map with size_order)
# YOUR CODE HERE
size_order = {'small': 0, 'medium' : 1, 'large': 2}
df['size_code'] = df['size'].map(size_order)
print(df[['size', 'size_code']])
# 3. Why one-hot is wrong for size / ordinal is wrong for city: ...   (comment)
print(" one hot is wrong for size as they are being classifies as the order they come as per the position in the vocabulary its not case over here. \n ordinal is wrong for city as city data is a nominal data and we simply cannot determine a city better or worse like newyork is better tha canada? we can't simply evalauate the data under oridinal ")

   delhi  mumbai  pune
0      0       0     1
1      1       0     0
2      0       1     0
3      0       0     1
4      1       0     0
5      0       1     0
6      0       0     1
7      1       0     0
     size  size_code
0   small          0
1   large          2
2  medium          1
3  medium          1
4   small          0
5   large          2
6   large          2
7   small          0
 one hot is wrong for size as they are being classifies as the order they come as per the position in the vocabulary its not case over here. 
 ordinal is wrong for city as city data is a nominal data and we simply cannot determine a city better or worse like newyork is better tha canada? we can't simply evalauate the data under oridinal 


#2. Feature scaling

In [23]:
# -----------------------------------------------------------
# 🔹 2A. THE PROBLEM — FEATURES ON DIFFERENT SCALES
# -----------------------------------------------------------

# income (tens of thousands) dwarfs age (tens). A distance-based
# model would treat income as nearly all that matters.
print(df[['age', 'income']].describe().loc[['min', 'max', 'mean']])

        age    income
min   22.00   38000.0
max   52.00  120000.0
mean  35.75   69625.0


In [28]:
# -----------------------------------------------------------
# 🔹 2B. STANDARDISATION (Z-score)  vs  NORMALISATION (Min-Max)
# -----------------------------------------------------------
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num = df[['age', 'income']]

z = StandardScaler().fit_transform(num)        # mean 0, std 1
m = MinMaxScaler().fit_transform(num)          # range [0, 1]

print('Standardised (mean~0, std~1):')
print(pd.DataFrame(z, columns=['age', 'income']).round(2).head(3))
print('\nMin-Max (range 0..1):')
print(pd.DataFrame(m, columns=['age', 'income']).round(2).head(3))

Standardised (mean~0, std~1):
    age  income
0 -1.10   -1.16
1  0.54    0.82
2 -0.28   -0.54

Min-Max (range 0..1):
    age  income
0  0.10    0.00
1  0.63    0.66
2  0.37    0.21


#### 🧪 LAB EXERCISE 2 — Scale a feature

Using the `income` column:
1. Standardise it with `StandardScaler` and confirm the result's mean is ~0 and std ~1.
2. Min-Max scale it and confirm the result's min is 0 and max is 1.
3. In a comment, name one model type that **needs** scaling and one that **doesn't**.

In [38]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

income = df[['income']]   # 2D shape for sklearn

# 1. standardise -> check mean ~0, std ~1
# YOUR CODE HERE
z = StandardScaler().fit_transform(income)
m = MinMaxScaler().fit_transform(income)
#print (m, z)
# 2. min-max scale -> check min 0, max 1
# YOUR CODE HERE
print(pd.DataFrame(z, columns=['income'] ))
# 3. Needs scaling: ...   Doesn't need it: ...   (comment)
print("KNN needs scaling , linear regression requires scaling whereas tree based model doesnt need any scaling.")


     income
0 -1.158468
1  0.819628
2 -0.535734
3 -0.828786
4  1.845307
5  0.233525
6  0.673102
7 -1.048574
KNN needs scaling , linear regression requires scaling whereas tree based model doesnt need any scaling.


#3. Avoiding data leakage

**Golden rule:** fit scalers/encoders on the **training set only**, then apply to the test set. Fitting on all the data lets test information leak in.

In [39]:
# -----------------------------------------------------------
# 🔹 3A. SPLIT FIRST, THEN FIT ON TRAIN ONLY
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['age', 'income']]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # FIT + transform on train
X_test_s  = scaler.transform(X_test)        # only TRANSFORM on test
print('train rows:', X_train.shape[0], '| test rows:', X_test.shape[0])
print('scaler learned mean from TRAIN only:', scaler.mean_.round(1))

train rows: 6 | test rows: 2
scaler learned mean from TRAIN only: [3.45e+01 6.90e+04]


#### 🧪 LAB EXERCISE 3 — Spot the leak

1. The two lines below show the **wrong** way (fit on all data) and the **right** way (fit on train). Run them and compare the learned means.
2. In a comment, explain why fitting on all the data before splitting is a problem.

In [42]:
# 1a. WRONG: fit on ALL data, then split (leakage!)
wrong_mean = StandardScaler().fit(X).mean_
print(wrong_mean)

# 1b. RIGHT: fit on TRAIN only
right_mean = StandardScaler().fit(X_train).mean_
print(right_mean)

print('fit-on-all mean :', wrong_mean.round(1))
print('fit-on-train mean:', right_mean.round(1))

# 2. Why fitting on all data is a problem: ...   (write your answer)
print("fitting on all data before splitting doesnt help us in understanding wheather our model is working the way we are excpecting we always need to split the data first and then fit on training data set itself. \n its always a best practise to split the data 80/20 or 70/30. ")

[3.5750e+01 6.9625e+04]
[3.45e+01 6.90e+04]
fit-on-all mean : [3.5800e+01 6.9625e+04]
fit-on-train mean: [3.45e+01 6.90e+04]
 fitting on all data before splitting doesnt help us in understanding wheather our model is working the way we are excpecting we always need to split the data first and then fit on training data set itself. 
 its always a best practise to split the data 80/20 or 70/30. 


#4. Feature engineering

In [43]:
# -----------------------------------------------------------
# 🔹 4A. COMBINE & BIN EXISTING COLUMNS
# -----------------------------------------------------------

fe = df.copy()
# combine: income per year of age (a crude 'earning rate')
fe['income_per_age'] = (fe['income'] / fe['age']).round(0)
# bin: turn continuous age into life-stage buckets
fe['age_group'] = pd.cut(fe['age'], bins=[0, 30, 45, 100],
                         labels=['young', 'mid', 'senior'])
fe[['age', 'income', 'income_per_age', 'age_group']]

,age,income,income_per_age,age_group
0,25,38000,1520.0,young
1,41,92000,2244.0,mid
2,33,55000,1667.0,mid
3,29,47000,1621.0,young
4,52,120000,2308.0,senior
5,38,76000,2000.0,mid
6,46,88000,1913.0,senior
7,22,41000,1864.0,young


In [44]:
# -----------------------------------------------------------
# 🔹 4B. EXTRACT FROM A DATE
# -----------------------------------------------------------
dates = pd.to_datetime(['2024-01-06', '2024-03-15', '2024-07-21', '2024-12-25'])
d = pd.DataFrame({'date': dates})
d['day_of_week'] = d['date'].dt.day_name()   # Monday, Tuesday, ...
d['month']       = d['date'].dt.month
d['is_weekend']  = d['date'].dt.dayofweek >= 5
d

,date,day_of_week,month,is_weekend
0,2024-01-06,Saturday,1,True
1,2024-03-15,Friday,3,False
2,2024-07-21,Sunday,7,True
3,2024-12-25,Wednesday,12,False


#### 🧪 LAB EXERCISE 4 — Engineer two features

On a fresh copy `ex = df.copy()`:
1. Create `high_earner` = `True` when `income` is above its median.
2. Bin `income` into 3 labelled buckets with `pd.cut` (e.g. low / mid / high).
3. Show the new columns next to `income`.

In [57]:
ex = df.copy()

# 1. high_earner flag (income > median)
# YOUR CODE HERE
fe['high_earner']= ex['income'] > ex['income'].median()
print(fe['high_earner'])

# 2. bin income into 3 buckets with pd.cut
# YOUR CODE HERE
fe['income'] = pd.cut(ex['income'], bins = 3, labels = ['low', 'mid', 'high'])
print(fe)
# 3. show income + the new columns
# YOUR CODE HERE
print(fe[['income', 'high_earner']])

0    False
1     True
2    False
3    False
4     True
5     True
6     True
7    False
Name: high_earner, dtype: bool
     city    size  age income  bought  size_code  size_order  income_per_age  \
0    pune   small   25    low       0          0           0          1520.0   
1   delhi   large   41    mid       1          2           2          2244.0   
2  mumbai  medium   33    low       0          1           1          1667.0   
3    pune  medium   29    low       0          1           1          1621.0   
4   delhi   small   52   high       1          0           0          2308.0   
5  mumbai   large   38    mid       1          2           2          2000.0   
6    pune   large   46    mid       1          2           2          1913.0   
7   delhi   small   22    low       0          0           0          1864.0   

  age_group  high_earner  
0     young        False  
1       mid         True  
2       mid        False  
3     young        False  
4    senior         True 

#5. The preprocessing pipeline

In [58]:
# -----------------------------------------------------------
# 🔹 5A. ONE LEAK-FREE PIPELINE: PREPROCESS + MODEL
# -----------------------------------------------------------
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_cols = ['age', 'income']
cat_cols = ['city', 'size']

# scale the numbers, one-hot the categories — all in one object
pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
pipe = Pipeline([('prep', pre), ('model', LogisticRegression(max_iter=1000))])
print(pipe)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'size'])])),
                ('model', LogisticRegression(max_iter=1000))])


In [59]:
# -----------------------------------------------------------
# 🔹 5B. FIT THE WHOLE THING IN ONE CALL
# -----------------------------------------------------------

X = df[num_cols + cat_cols]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

pipe.fit(X_train, y_train)        # preprocessing fitted on TRAIN only — no leakage
acc = pipe.score(X_test, y_test)  # transforms test with train-fitted steps
print('Test accuracy:', round(acc, 2))
print('(small toy dataset — the point is the leak-free workflow, not the score)')

Test accuracy: 1.0
(small toy dataset — the point is the leak-free workflow, not the score)


#### 🧪 LAB EXERCISE 5 — Build your own pipeline

1. Build a `Pipeline` of just `MinMaxScaler` + `LogisticRegression` on the numeric columns (`age`, `income`).
2. Split the data, then `fit` on the training set.
3. Print the test accuracy with `.score(...)`.

In [60]:
from sklearn.preprocessing import MinMaxScaler

Xn = df[['age', 'income']]
yn = df['bought']

# 1. Pipeline: MinMaxScaler -> LogisticRegression
# YOUR CODE HERE
pipe = Pipeline([('scaler', MinMaxScaler()), ('model', LogisticRegression())])


# 2. split + fit on train
# YOUR CODE HERE
X_train, X_test, y_train, y_test = train_test_split(
    Xn, yn, test_size=0.25, random_state=0)

pipe.fit(X_train, y_train)


# 3. print test accuracy
# YOUR CODE HERE
print(pipe.score(X_test, y_test))


1.0


#📘 Summary — Preprocessing toolkit

| Step | When | Key calls |
| ---- | ---- | --------- |
| **One-hot encode** | unordered categories | `pd.get_dummies` · `OneHotEncoder` |
| **Ordinal encode** | ordered categories | `.map({...})` · `OrdinalEncoder` |
| **Standardise** | most models (default) | `StandardScaler` |
| **Min-Max scale** | bounded data, images | `MinMaxScaler` |
| **Avoid leakage** | always | `fit` on train, `transform` on test |
| **Engineer features** | expose the signal | `pd.cut` · combine columns · `.dt` |
| **Pipeline** | tie it together | `Pipeline` · `ColumnTransformer` |

**Homework (self-paced):** one-hot an unordered column · ordinal-encode an ordered one · standardise & min-max scale a feature · engineer 2 features · build a scaler + model Pipeline.

**U8 complete — data is clean & model-ready.** Next: **U9 Exploratory Data Analysis** — seeing the patterns in your data before modelling.